# Notebook zur Umwandlung der Prüfungsordnungen in Markdown-Dateien

### Bibliotheken installieren

In [ ]:
!pip install docling

In [ ]:
!pip install -U transformers

### Docling importieren

In [ ]:
from docling.document_converter import DocumentConverter, FormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

### PDF-Pipeline und DocumentConverter initialisieren

In [ ]:
pipeline_options = PdfPipelineOptions(do_table_structure=True)
pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE  # use more accurate TableFormer model
pipeline_options.do_ocr = False

# Create a document converter with specified format options
converter = DocumentConverter(
    allowed_formats=[
            InputFormat.PDF,
            InputFormat.IMAGE,
        ],  # whitelist formats, non-matching files are ignored.
    format_options={
        InputFormat.PDF: FormatOption(
            pipeline_cls=StandardPdfPipeline,
            pipeline_options=pipeline_options,
            backend=DoclingParseV4DocumentBackend)
    }
)

### Umwandlung der Prüfungsordnungen in Markdown-Dateien

Die PDF-Dateien aus dem Ordner `pruefungsordnungen` werden mit der Funktion `pdf_to_markdown(input_path, output_path)` in Markdown-Dateien umgewandelt und im Ordner `pruefungsordnungenMarkdown` gespeichert. Für die Umwandlung wird die Bibliothek Docling verwendet.

In [ ]:
import os

input_base_folder = "pruefungsordnungen/"
output_base_folder = "pruefungsordnungenMarkdown/"

def pdf_to_markdown(input_path, output_path):
    result = converter.convert(input_path)
    docling_doc = result.document
    markdown = docling_doc.export_to_markdown()
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown)
    

# Durchlaufe alle Ordner und Dateien unter input_base_folder
for root, dirs, files in os.walk(input_base_folder):
    print(root)
    for datei in files:
        input_file = os.path.join(root, datei)
        
        # Ersetze input_base_folder durch output_base_folder im Pfad
        output_folder = root.replace(input_base_folder, output_base_folder, 1)
        
        # Zielordner erstellen, falls nicht vorhanden
        os.makedirs(output_folder, exist_ok=True)
        
        # Bestimme neuen Dateinamen basierend auf dem Namen der Originaldatei
        ziel_datei = f"{datei[:-4]}.md"
        
        output_file = os.path.join(output_folder, ziel_datei)
        
        # Datei verarbeiten und Ergebnis speichern
        pdf_to_markdown(input_file, output_file)



### Markdown-Dateien in Ordner zippen

In [ ]:
import shutil

zip_name = 'pruefungsordnungenMd'

shutil.make_archive(zip_name, 'zip', output_base_folder)